# latin-mv-tlt — train the v0.2 translation model

Replaces `colab_train_realize.ipynb`. Two structural differences:

- **one** training run, not two. v0.2 uses a single model for both directions,
  selected by a task prefix (R-3.1).
- an **export cell**. v0.1 had none, so the ONNX conversion happened outside any
  committed tooling — which is how 307 MB of models ended up containing ~162 MB
  of graphs the runtime never loads. The size gate (R-3.4) now runs here, at M-4,
  before anything is integrated.

Before you start, on your laptop (not here — the corpus builder needs Node for
the transliterator, R-2.2):

```
python tools/build_translation_pairs.py          # M-1
python tools/measure_roundtrip.py --n 1000       # M-2   gate: latin-stable >= 98%
python tools/profile_tokenizer.py                # M-2b  gate: <unk> <= 5%
```

Upload `train.jsonl` and `valid.jsonl` when prompted below.


## 1. GPU


In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU.'
print(torch.cuda.get_device_name(0))


Tesla T4


## 2. Dependencies

Mirrors `tools/requirements.txt`. `transformers>=4.46` is required for
`processing_class` on the trainer.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
%pip install -q -U 'transformers>=4.46' 'accelerate>=1.1' sentencepiece \
    'sacrebleu>=2.4' 'optimum[onnxruntime]>=1.23' 'onnx>=1.17' 'onnxruntime>=1.19'

# Colab preinstalls a `diffusers` that is incompatible with its `huggingface_hub`
# (`cannot import name 'get_cached_repo_tree'`). optimum imports diffusers behind
# `is_diffusers_available()`, so that breakage takes down the ONNX export of a
# seq2seq model that has nothing to do with diffusion. Nothing here uses it.
%pip uninstall -y -q diffusers



## 3. Get the code and the corpus

The training and export scripts are committed, so they are cloned rather than
pasted — a notebook copy of the trainer is a second implementation to keep in
sync, which is exactly what went wrong in v0.1 (`FrameDataset` existed verbatim
in both the script and the notebook).


In [17]:
import os
REPO = 'https://github.com/Wildeys/latin-mv-tlt.git'   # adjust if your remote differs
if not os.path.exists('latin-mv-tlt'):
    !git clone --depth 1 $REPO
%cd latin-mv-tlt
!mkdir -p data/parallel


Cloning into 'latin-mv-tlt'...
remote: Enumerating objects: 166, done.
remote: Counting objects: 100% (166/166), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 166 (delta 10), reused 103 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (166/166), 63.42 MiB | 19.95 MiB/s, done.
Resolving deltas: 100% (10/10), done.
Updating files: 100% (138/138), done.
/content/latin-mv-tlt/latin-mv-tlt/latin-mv-tlt/latin-mv-tlt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Upload train.jsonl and valid_small.jsonl to Drive from your browser once.
# files.upload() stalls or dies at 189 MB, and a dropped session would make you
# do it again. valid_small.jsonl is the 2,000-row subset from TRAINING.md step 3
# — generating over all 49,948 valid rows costs 30-50 min per evaluation.
!cp /content/drive/MyDrive/training_datasets/train.jsonl /content/drive/MyDrive/training_datasets/valid.jsonl data/parallel/
!ls -la data/parallel/


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 211216
drwxr-xr-x 2 root root      4096 Aug 27 10:50 .
drwxr-xr-x 3 root root      4096 Aug 27 10:48 ..
-rw-r--r-- 1 root root      2707 Aug 27 10:48 corpus_stats.json
-rw------- 1 root root 198250766 Aug 27 10:50 train.jsonl
-rw------- 1 root root  18013937 Aug 27 10:50 valid.jsonl


In [ ]:
import json, random
rows = [json.loads(l) for l in open('data/parallel/valid.jsonl', encoding='utf-8') if l.strip()]
random.Random(11).shuffle(rows)
keep = {'dv-en': [], 'en-dv': []}
for r in rows:
    if len(keep[r['direction']]) < 1000:
        keep[r['direction']].append(r)
out = '/content/drive/MyDrive/valid_small.jsonl'
with open(out, 'w', encoding='utf-8') as f:
    for r in keep['dv-en'] + keep['en-dv']:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')
print(out, sum(len(v) for v in keep.values()), 'rows')

/content/drive/MyDrive/valid_small.jsonl 2000 rows


## 4. Verify the corpus before training

v0.1's notebook asserted hard-coded line counts, which broke every time the
corpus legitimately changed. These checks assert *properties* instead: the
prefix is present, both directions are represented, and no Thaana reached the
model input — that last one is the architecture's whole premise (§1.1).


In [ ]:
import json, re, collections

def load(path):
    return [json.loads(l) for l in open(path, encoding='utf-8') if l.strip()]

train = load('data/parallel/train.jsonl')
valid = load('/content/drive/MyDrive/valid_small.jsonl')
THAANA = re.compile('[\u0780-\u07BF]')

for name, rows in (('train', train), ('valid', valid)):
    assert rows, f'{name} is empty'
    dirs = collections.Counter(r['direction'] for r in rows)
    assert set(dirs) == {'dv-en', 'en-dv'}, f'{name}: expected both directions, got {dict(dirs)}'
    for r in rows:
        assert r['input'].startswith('translate '), f'unprefixed row: {r["input"][:60]}'
        assert not THAANA.search(r['input']), 'Thaana reached the model input'
    print(f'{name:<6} {len(rows):>7} rows  {dict(dirs)}')

train_inputs = {r['input'] for r in train}
leaked = sum(1 for r in valid if r['input'] in train_inputs)
assert leaked == 0, f'{leaked} valid inputs also appear in train (R-2.6)'
print('no train/valid leakage')


train   480018 rows  {'dv-en': 240009, 'en-dv': 240009}
valid     2000 rows  {'dv-en': 1000, 'en-dv': 1000}
no train/valid leakage


## 5. Train (M-3)

`tools/train_translate.py` carries the hyperparameters from R-9.2: **lr 1e-4**
(not v0.1's 5e-5, and not the 3e-4 the spec first suggested), weight decay 0.01,
batch 32, and `load_best_model_at_end` on chrF++ — without which `save_model()`
keeps the *last* epoch rather than the best.

`SAVE_STEPS` and `RESUME` are what make a dropped session survivable. Saving
once per epoch is ~15,000 steps at batch 32, so a timeout at step 50,000 throws
away everything since 45,003; `SAVE_STEPS = 5000` bounds that. `RESUME = 'auto'`
takes the newest checkpoint in `<OUT>/runs`, and **fails** if there is none
rather than quietly spending four hours training from scratch because Drive was
not mounted. Set it to `None` when you mean to start over.

Set `SMOKE = True` for a 30-second wiring check. R-8.3: a smoke checkpoint must
never be demoed or evaluated as a result.


### Checkpoints live in Drive

`OUT` is a Drive path, not a directory in the clone. Colab's own disk dies with
the session, so a checkpoint written under `latin-mv-tlt/` is gone the moment the
tab drops — and at ~730 MB each that is hours of GPU time.

Resuming needs **every** checkpoint of the interrupted run in `<OUT>/runs`, not
just the newest. `trainer_state.json` records the best checkpoint as an absolute
path from the session that wrote it; transformers rebuilds that path under
`output_dir` on each save, and when the directory is missing it only *warns* —
then `save_model()` ships the last step instead of the best one.
`train_translate.py` refuses to start rather than let that happen quietly.

Run this to see what the trainer can reach, and `!mv` anything stranded
elsewhere in Drive into `runs/`.


In [ ]:
import glob, os
OUT = '/content/drive/MyDrive/training_datasets/dv-en-translate-checkpoints'
RUNS = f'{OUT}/runs'
os.makedirs(RUNS, exist_ok=True)

print('the trainer can resume from:')
for d in sorted(glob.glob(f'{RUNS}/checkpoint-*')):
    print('  ', os.path.basename(d))

stranded = {d for pat in ('/content/drive/MyDrive/training_datasets/dv-en-translate-checkpoints/runs/checkpoint-*',
                          '/content/drive/MyDrive/*/checkpoint-*',
                          '/content/drive/MyDrive/*/*/checkpoint-*')
            for d in glob.glob(pat) if not d.startswith(RUNS)}
if stranded:
    print('\nelsewhere in Drive — move these in if they belong to this run:')
    for d in sorted(stranded):
        print('  ', d)
    print(f"\n  !mv '<path>' '{RUNS}/'")


the trainer can resume from:
   checkpoint-30002
   checkpoint-45003


In [ ]:
SMOKE = False
BATCH = 32          # drop to 8-16 on CUDA OOM, or for flan-t5-base
SAVE_STEPS = 5000   # ~15,000 steps per epoch, so an interruption costs <= 5,000
RESUME = 'auto'     # None for a fresh run; 'auto' after a dropped session

!python tools/train_translate.py \
    --train data/parallel/train.jsonl \
    --valid /content/drive/MyDrive/valid_small.jsonl \
    --out {OUT} \
    --model t5-small --epochs 4 --batch {BATCH} --lr 1e-4 \
    --save-steps {SAVE_STEPS} \
    {'--resume ' + RESUME if RESUME else ''} \
    {'--smoke' if SMOKE else ''}


2026-08-27 11:10:05.353374: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
resuming checkpoint-45003: step 45003/60004, epoch 3.0
best so far: checkpoint-45003, chrf++ 33.289
There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].
	eval_steps: 5000 (from args) != 500 (from trainer_state.json)
	save_steps: 5000 (from args) != 500 (from trainer_state.json)
{'loss': 1.7492, 'grad_norm': 1.2401765584945679, 'learning_rate': 2.4840010665955606e-05, 'epoch': 3.01}
{'loss': 1.7352, 'grad_norm': 1.3959736824035645, 'learning_rate': 2.467335510965936e-05, 'epoch': 3.01}
{'loss': 1.7453, 'grad_norm': 1.348825454711914, 'learning_rate': 2.4506699553363112e-05, 'epoch': 3.02}


### Per-epoch metrics

R-8.5. Record this table for the write-up: it is the evidence that the best
checkpoint was selected on a metric rather than on the last epoch.


In [17]:
OUT = '/content/drive/MyDrive/training_datasets/dv-en-translate-checkpoints'
import json
stats = json.load(open(f'{OUT}/training_stats.json'))
for row in stats['perEpoch']:
    print({k: v for k, v in row.items() if k.startswith('eval_') or k == 'epoch'})
print('\nbest chrF++:', stats['bestMetric'])


{'epoch': 1.0, 'eval_bleu': 3.2739, 'eval_bleu_dv-en': 3.9483, 'eval_bleu_en-dv': 2.5807, 'eval_chrf': 23.5988, 'eval_chrf_dv-en': 21.9802, 'eval_chrf_en-dv': 24.7156, 'eval_loss': 1.9933170080184937, 'eval_runtime': 2731.3664, 'eval_samples_per_second': 18.287, 'eval_steps_per_second': 0.572}
{'epoch': 2.0, 'eval_bleu': 5.5889, 'eval_bleu_dv-en': 6.2369, 'eval_bleu_en-dv': 4.7617, 'eval_chrf': 30.5842, 'eval_chrf_dv-en': 27.7192, 'eval_chrf_en-dv': 32.6854, 'eval_loss': 1.6507021188735962, 'eval_runtime': 2302.7298, 'eval_samples_per_second': 21.691, 'eval_steps_per_second': 0.678}
{'epoch': 3.0, 'eval_bleu': 6.9665, 'eval_bleu_dv-en': 7.8236, 'eval_bleu_en-dv': 5.8947, 'eval_chrf': 33.289, 'eval_chrf_dv-en': 29.9388, 'eval_chrf_en-dv': 35.7766, 'eval_loss': 1.520668387413025, 'eval_runtime': 2103.6742, 'eval_samples_per_second': 23.743, 'eval_steps_per_second': 0.742}
{'eval_loss': 1.7816272974014282, 'eval_bleu': 7.1646, 'eval_chrf': 33.4131, 'eval_bleu_dv-en': 8.3242, 'eval_chrf_dv

## 6. Probe


In [4]:
OUT = '/content/drive/MyDrive/training_datasets/dv-en-translate-checkpoints'
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tok = AutoTokenizer.from_pretrained(OUT)
mdl = AutoModelForSeq2SeqLM.from_pretrained(OUT).cuda().eval()

def go(text):
    ids = tok(text, return_tensors='pt').input_ids.cuda()
    out = mdl.generate(ids, max_new_tokens=128, num_beams=1, do_sample=False)
    return tok.decode(out[0], skip_special_tokens=True)

print(go('translate Dhivehi Latin to English: aharen maleah dhaanan maadhama. miadhu nudhaanan'))
print(go('translate English to Dhivehi Latin: I will go to Male tomorrow, but not today'))


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

I'll go to Male tomorrow. I won't go today
maadhamaa maale dhaanan, ekamaku miadhu nudhaanan


In [5]:
!cp /content/drive/MyDrive/training_datasets/valid.jsonl /content/drive/MyDrive/training_datasets/test.jsonl /content/drive/MyDrive/training_datasets/train.jsonl /content/drive/MyDrive/test.jsonl data/parallel/ 2>/dev/null
!ls -la data/parallel/*.jsonl

TRIMMED = OUT + '-trimmed'
!python tools/trim_vocab.py \
    --model {OUT} \
    --corpus data/parallel/train.jsonl data/parallel/valid.jsonl data/parallel/test.jsonl \
    --out {TRIMMED}


-rw------- 1 root root  15173881 Aug 27 18:39 data/parallel/test.jsonl
-rw------- 1 root root 198250766 Aug 27 18:39 data/parallel/train.jsonl
-rw------- 1 root root  18013937 Aug 27 18:39 data/parallel/valid.jsonl
Loading weights: 100% 131/131 [00:00<00:00, 1852.24it/s]
1. loaded /content/drive/MyDrive/training_datasets/dv-en-translate-checkpoints
  tokenizer vocab   32,100
  embedding rows    32,128
  unreachable rows  28 (padding, dropped)

2. scan corpus (3 files)
  570,558 rows, 1,141,116 texts
      100,000 / 1,141,116 texts, 12,246 ids so far
      200,000 / 1,141,116 texts, 17,696 ids so far
      300,000 / 1,141,116 texts, 20,535 ids so far
      400,000 / 1,141,116 texts, 21,482 ids so far
      500,000 / 1,141,116 texts, 22,037 ids so far
      600,000 / 1,141,116 texts, 22,403 ids so far
      700,000 / 1,141,116 texts, 22,641 ids so far
      800,000 / 1,141,116 texts, 22,859 ids so far
      900,000 / 1,141,116 texts, 23,109 ids so far
    1,000,000 / 1,141,116 texts, 23,

## 7. Export to ONNX INT8 (M-4)

This is the cell v0.1 never had. `tools/export_onnx.py` asserts at every step:

- the merged decoder really exposes `use_cache_branch` — the assertion that makes
  deleting the old `runBeam` monkey-patch safe;
- each quantized graph actually shrank, which catches `quantize_dynamic` silently
  skipping `If` subgraphs and leaving the file fp32 inside;
- the total is within the 80 MB budget, and it **refuses to write** if not.

If the budget gate fails, the contingency ladder is in the error message and in
REQUIREMENTS.md R-3.2. Vocabulary trimming is the big win and has to happen
*before* retraining, so do not skip past this.


In [16]:
OUT = '/content/drive/MyDrive/training_datasets/dv-en-translate-checkpoints-trimmed'
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tok = AutoTokenizer.from_pretrained(OUT)
mdl = AutoModelForSeq2SeqLM.from_pretrained(OUT).cuda().eval()

def go(text):
    ids = tok(text, return_tensors='pt').input_ids.cuda()
    out = mdl.generate(ids, max_new_tokens=128, num_beams=1, do_sample=False)
    return tok.decode(out[0], skip_special_tokens=True)

print(go('translate Dhivehi Latin to English: aharen maleah dhaanan maadhama. miadhu nudhaanan'))
print(go('translate English to Dhivehi Latin: I will go to Male tomorrow, but not today'))


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

I'll go to Male tomorrow. I won't go today
maadhamaa maale dhaanan, ekamaku miadhu nudhaanan


In [19]:
import sys

TRIMMED = '/content/drive/MyDrive/training_datasets/dv-en-translate-checkpoints-trimmed'
# `!{sys.executable}`, not `!python`: %pip installed optimum into the kernel's
# interpreter, and a bare `python` may not be that interpreter.
!{sys.executable} tools/export_onnx.py \
    --model {TRIMMED} \
    --out public/models/dv-en-translate



1. optimum-cli export
2026-08-27 19:05:08.412940: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.13/dist-packages/optimum/exporters/base.py:151: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  self._normalized_config = self.NORMALIZED_CONFIG_CLASS(self._config)
Opset 14 is lower than the recommended minimum opset (18) to export t5. The ONNX export may fail or the exported model may be suboptimal.
/usr/local/lib/python3.13/dist-packages/transformers/models/t5/modeling_t5.py:1271: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorr

In [20]:
import json
print(json.dumps(json.load(open('public/models/dv-en-translate/export_stats.json')), indent=2))


{
  "generatedBy": "tools/export_onnx.py",
  "sourceCheckpoint": "/content/drive/MyDrive/training_datasets/dv-en-translate-checkpoints-trimmed",
  "opset": 14,
  "quantization": {
    "weightType": "QInt8",
    "perChannel": false,
    "reduceRange": true,
    "enableSubgraph": true
  },
  "mergedDecoderHasCacheBranch": true,
  "files": {
    "config.json": 1498,
    "generation_config.json": 122,
    "tokenizer.json": 1855942,
    "tokenizer_config.json": 446,
    "onnx/encoder_model_quantized.onnx": 31100336,
    "onnx/decoder_model_merged_quantized.onnx": 37889196
  },
  "totalBytes": 70847540,
  "budgetBytes": 80000000,
  "withinBudget": true,
  "notShipped": [
    "decoder_model.onnx",
    "decoder_model_merged.onnx",
    "decoder_with_past_model.onnx",
    "encoder_model.onnx"
  ],
  "note": "Only the graphs transformers.js loads are shipped (R-3.13). v0.1 also carried decoder_with_past (never loaded) and an unmerged decoder that was byte-identical to the file served as the merge

## 8. Download

Unzip into `public/models/dv-en-translate/` in your working tree, then:

```
node tools/smoke_translate.mjs 'aharen maleah dhaanan'   # runs the export under Node
npm run check:models                                     # the same budget gate, in CI
npm run dev                                              # the only check that exercises WASM
```

Only after that is the v0.2 model verified, and only then does M-8b delete the
v0.1 realization models.


In [21]:
import shutil
from google.colab import files
shutil.make_archive('dv-en-translate', 'zip', 'public/models/dv-en-translate')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copy('dv-en-translate.zip', '/content/drive/MyDrive/')
    print('copied to Drive')
except Exception as exc:
    print('Drive copy skipped:', exc)
files.download('dv-en-translate.zip')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
copied to Drive


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>